# Phase 3 — preprocessing

Run the cells in order with Internet enabled, Accelerator set to None, and Persistence set to Files only. Keep `RUN_MODE = "smoke"` until the smoke results are approved.

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import sys
from datetime import UTC, datetime
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError("Select the Kaggle Jupyter Server kernel before running Phase 3.")

RUN_MODE = "smoke"  # Change to "full" only after the smoke run passes review.
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
RECORD_LIMIT = 100 if RUN_MODE == "smoke" else None
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
SCRATCH_ROOT = Path("/kaggle/temp/text-summarizer")
OUTPUT_DIRECTORY = KAGGLE_WORKING / "text-summarizer" / "phase3" / f"{RUN_MODE}-{RUN_ID}"
HF_HOME = SCRATCH_ROOT / "huggingface"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

disk = shutil.disk_usage(KAGGLE_WORKING)
print(f"Run mode: {RUN_MODE}")
print(f"Per-dataset/split record limit: {RECORD_LIMIT}")
print(f"Python: {platform.python_version()}")
print(f"Free working disk: {disk.free / (1024**3):.2f} GiB")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
import subprocess
from importlib.metadata import PackageNotFoundError, version

PROJECT_REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_BRANCH = "refactoring/phase-transformer"
PROJECT_SOURCE = SCRATCH_ROOT / "project"

if not (PROJECT_SOURCE / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            PROJECT_BRANCH,
            PROJECT_REPOSITORY,
            str(PROJECT_SOURCE),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(PROJECT_SOURCE), "pull", "--ff-only", "origin", PROJECT_BRANCH],
        check=True,
    )

project_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_SOURCE), "rev-parse", "HEAD"],
    text=True,
).strip()
sys.path.insert(0, str(PROJECT_SOURCE / "ml" / "src"))

required_packages = ("datasets", "kagglehub", "pydantic", "transformers")
missing_packages = []
for package_name in required_packages:
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        missing_packages.append(package_name)
if missing_packages:
    raise RuntimeError(f"Missing Kaggle packages: {missing_packages}")
print(f"Project commit: {project_commit}")

In [ ]:
from datasets import load_dataset

cnn_dailymail = load_dataset("abisee/cnn_dailymail", "3.0.0")
govreport = load_dataset("ccdv/govreport-summarization")
billsum = load_dataset("FiscalNote/billsum")

for dataset_name, dataset in (
    ("cnn_dailymail", cnn_dailymail),
    ("govreport", govreport),
    ("billsum", billsum),
):
    counts = {split: len(rows) for split, rows in dataset.items()}
    print(f"{dataset_name}: {counts}")

In [ ]:
import kagglehub


def clone_dataset(repository: str, destination: Path) -> Path:
    if not (destination / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", repository, str(destination)],
            check=True,
        )
    return destination


qmsum_root = clone_dataset("https://github.com/Yale-LILY/QMSum.git", SCRATCH_ROOT / "qmsum")
tweetsumm_root = clone_dataset("https://github.com/guyfe/Tweetsumm.git", SCRATCH_ROOT / "tweetsumm")
twcs_candidates = sorted(KAGGLE_INPUT.rglob("twcs.csv"))
if not twcs_candidates:
    twcs_download_root = Path(
        kagglehub.dataset_download(
            "thoughtvector/customer-support-on-twitter",
            output_dir=str(SCRATCH_ROOT / "customer-support-on-twitter"),
        )
    )
    twcs_candidates = sorted(twcs_download_root.rglob("twcs.csv"))
if len(twcs_candidates) != 1:
    raise RuntimeError(f"Expected one twcs.csv file, found: {twcs_candidates}")
twcs_csv_path = twcs_candidates[0]
print(f"QMSum: {qmsum_root}")
print(f"TweetSumm: {tweetsumm_root}")
print(f"Customer support source: {twcs_csv_path}")

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    cache_dir=HF_HOME / "models",
    use_fast=True,
)
print(f"Tokenizer: {TOKENIZER_ID}")
print(f"Fast tokenizer: {tokenizer.is_fast}")

In [ ]:
from summarizer_ml.preprocessing import run_phase3_diagnostic


def report_preprocessing_progress(processed: int) -> None:
    print(f"Preprocessed records: {processed:,}")


def report_token_progress(processed: int) -> None:
    print(f"Token-analyzed records: {processed:,}")


result = run_phase3_diagnostic(
    cnn_dailymail=cnn_dailymail,
    govreport=govreport,
    billsum=billsum,
    qmsum_root=qmsum_root,
    tweetsumm_root=tweetsumm_root,
    twcs_csv_path=twcs_csv_path,
    tokenizer=tokenizer,
    tokenizer_id=TOKENIZER_ID,
    output_directory=OUTPUT_DIRECTORY,
    max_records_per_dataset_split=RECORD_LIMIT,
    progress_every=100 if RUN_MODE == "smoke" else 10_000,
    progress_callback=report_preprocessing_progress,
    token_batch_size=256,
    token_progress_every=100 if RUN_MODE == "smoke" else 10_000,
    token_progress_callback=report_token_progress,
)
print(result.processing.model_dump_json(indent=2))
print(result.token_analysis.model_dump_json(indent=2))

In [ ]:
expected_datasets = {"cnn_dailymail", "govreport", "billsum", "qmsum", "tweetsumm"}
processed_datasets = set(result.processing.accepted_by_dataset)
if processed_datasets != expected_datasets:
    raise RuntimeError(
        f"Dataset coverage mismatch: expected {expected_datasets}, got {processed_datasets}"
    )
if result.processing.accepted_records != result.token_analysis.overall.input_tokens.count:
    raise RuntimeError("Processed and token-analyzed record counts do not match.")
if result.processing.accepted_records == 0:
    raise RuntimeError("Phase 3 accepted no records.")
required_reports = (
    OUTPUT_DIRECTORY / "processing-report.json",
    OUTPUT_DIRECTORY / "analysis" / "token-analysis-report.json",
)
missing_reports = [str(path) for path in required_reports if not path.is_file()]
if missing_reports:
    raise RuntimeError(f"Missing Phase 3 reports: {missing_reports}")

rejection_rate = result.processing.rejected_records / result.processing.input_records
print(f"Acceptance: {result.processing.accepted_records:,}/{result.processing.input_records:,}")
print(f"Rejection rate: {rejection_rate:.2%}")
print(f"Exact duplicates removed: {result.processing.exact_duplicates:,}")
print(f"Cross-split findings to review: {result.processing.leakage_findings:,}")
print(f"Artifacts: {OUTPUT_DIRECTORY}")
print(f"Phase 3 {RUN_MODE} diagnostic gate passed.")

In [ ]:
from summarizer_ml.preprocessing.finalization import (
    materialize_leakage_safe_partitions,
)

finalization_result = materialize_leakage_safe_partitions(OUTPUT_DIRECTORY)
if finalization_result.report.leakage_findings_after != 0:
    raise RuntimeError("Final partitions still contain cross-split leakage.")
if finalization_result.report.final_records != finalization_result.report.final_token_records:
    raise RuntimeError("Final partition and token-length counts do not match.")
print(finalization_result.report.model_dump_json(indent=2))
print(f"Final artifacts: {finalization_result.final_directory}")
print("Phase 3 leakage-safe finalization gate passed.")

In [ ]:
from summarizer_ml.preprocessing.outliers import audit_outliers


def report_audit_progress(stage: str, count: int) -> None:
    print(f"Outlier audit {stage}: {count:,}")


outlier_audit = audit_outliers(
    finalization_result.final_directory,
    progress_every=50_000,
    progress_callback=report_audit_progress,
)
if outlier_audit.report.total_processed_records != outlier_audit.report.total_token_records:
    raise RuntimeError("Outlier audit record counts do not match.")
print(outlier_audit.report.model_dump_json(indent=2))
print(f"Outlier audit artifacts: {outlier_audit.output_directory}")
print("Phase 3 outlier audit gate passed.")